In [ ]:
%pip install -U -q numpy pandas wfdb matplotlib notebook scikit-learn PyWavelets plotly

In [ ]:
%pip install matplotlib

In [2]:
# %load_ext autoreload
# %autoreload 2

import wfdb
#import matplotlib.pyplot as plt
import numpy as np
import os
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression

from funkcje import *
from processor import Signal, ICP_ABP_Processor

## Sekcja wczytwania


In [3]:
%reload_ext autoreload
data_dir = 'charis-database-1.0.0'  # Lokalny folder
record_name = 'charis1'     # Nazwa rekordu (.hea i .dat)

os.makedirs(data_dir, exist_ok=True)

# TERAZ TWÓJ KOD BEZ ZMIAN
local_path = os.path.join(data_dir, record_name)

# Odczyt (automatycznie znajdzie .hea i .dat)
record = wfdb.rdrecord(local_path)

# Dane sygnałów
signals = record.p_signal  # NxM tablica (próbki x kanały)
fs = record.fs  # Częstotliwość próbkowania
time = np.arange(len(signals)) / fs # sekundy

print(f"Rekord: {record_name}")
print(f"Długość: {len(signals)} próbek ({len(signals)/fs/3600:.1f} h)")
print(f"fs: {fs} Hz, Kanały: {record.n_sig}")
print(f"Nazwy kanałów: {record.sig_name}")  # ['ABP', 'ECG', 'ICP']


channel_ABP = signals[:,0]
channel_ICP = signals[:,2]

# Inicjalizacja
processor = ICP_ABP_Processor(channel_ICP, channel_ABP, sampling_freq=50)
processor.process_all()


Rekord: charis1
Długość: 12239851 próbek (68.0 h)
fs: 50 Hz, Kanały: 3
Nazwy kanałów: ['ABP', 'ECG', 'ICP']
[ICP] Szumy usunięte.
[ABP] Szumy usunięte.
[ICP] Sygnał przefiltrowany.
[ABP] Sygnał przefiltrowany.


In [ ]:
# Wizualizacja pierwszych 60s
plt.figure(figsize=(15, 6))
plt.plot(time[int(6000*fs):int(12000*fs)], processor.icp.data[int(6000*fs):int(12000*fs)], linewidth=0.8)
plt.xlabel('Czas [s]')
plt.ylabel('Amplituda [mmHg]')
plt.title(f'{record_name} kanał ICP')
plt.grid(True, alpha=0.3)
plt.show()

# Wizualizacja pierwszych 60s
plt.figure(figsize=(15, 6))
plt.plot(time[int(6000*fs):int(12000*fs)], processor.abp.data[int(6000*fs):int(12000*fs)], linewidth=0.8)
plt.xlabel('Czas [s]')
plt.ylabel('Amplituda [mmHg]')
plt.title(f'{record_name} kanał ABP')
plt.grid(True, alpha=0.3)
plt.show()

In [4]:
x_seconds_avg = 5
window_size = 60

okna_icp, okna_abp = processor.get_windowed_data(x_seconds_avg=x_seconds_avg, window_size=window_size)
wyniki_prx = calculate_PRx(okna_icp, okna_abp)

liczba_okien = len(wyniki_prx)

czas_start_minuty = (window_size * x_seconds_avg) / 60  # Dla 60 i 5 to będzie równo 5.0 min

# Krok czasowy między kolejnymi oknami w minutach
krok_minuty = x_seconds_avg / 60  # 5 / 60 = 0.08333...

# 4. Generowanie wektora osi X
# Oś X zaczyna się od czasu_start i trwa przez tyle kroków, ile mamy wyników PRx
czas_os_x_minuty = czas_start_minuty + np.arange(liczba_okien) * krok_minuty

# --- 5. RYSUJ INTERAKTYWNY WYKRES (ZAMIAST MATPLOTLIB) ---
fig = go.Figure()

# Dodanie serii danych PRx ze zdefiniowaną osią czasu w minutach
fig.add_trace(go.Scatter(
    x=czas_os_x_minuty,
    y=wyniki_prx,
    mode='lines',
    name='PRx (Korelacja ICP/ABP)',
    line=dict(color='blue', width=2),
    # Precyzyjne formatowanie dymka (hover) po najechaniu myszką
    hovertemplate='<b>Czas:</b> %{x:.2f} min<br><b>PRx:</b> %{y:.3f}<extra></extra>'
))

# Konfiguracja osi, siatki oraz interakcji (zoom/pan)
fig.update_layout(
    title=dict(
        text='Monitorowanie indeksu PRx w czasie',
        x=0.5,
        font=dict(size=16)
    ),
    xaxis=dict(
        title='Czas (minuty)',
        gridcolor='rgba(200, 200, 200, 0.2)',
        showspikes=True,       # Pionowa linia pomocnicza śledząca kursor
        spikemode='across',
        spikethickness=1,
        spikedash='dash'
    ),
    yaxis=dict(
        title='Wartość PRx',
        range=[-1.1, 1.1],     # Sztywny zakres dla współczynnika korelacji Pearsona
        gridcolor='rgba(200, 200, 200, 0.2)',
        zeroline=True,
        zerolinecolor='rgba(0, 0, 0, 0.3)',
        zerolinewidth=1
    ),
    template='plotly_white',
    hovermode='x unified',     # Łączenie etykiet danych dla tej samej wartości osi X
    dragmode='zoom',           # Domyślne działanie myszy to zaznaczanie obszaru (zoom)
    width=1600,                 # Odpowiednik figsize=(10, 4)
    height=400
)

# Wyświetlenie wykresu (w Jupyter Notebook pojawi się w komórce, w skrypcie .py otworzy przeglądarkę)
fig.show()

In [ ]:
x_seconds_avg = 10
window_size = 30

okna_icp, okna_abp = processor.get_windowed_data(x_seconds_avg=x_seconds_avg, window_size=window_size)
wyniki_prx = calculate_PRx(okna_icp, okna_abp)

liczba_okien = len(wyniki_prx)

czas_start_minuty = (window_size * x_seconds_avg) / 60  # Dla 60 i 5 to będzie równo 5.0 min

# Krok czasowy między kolejnymi oknami w minutach
krok_minuty = x_seconds_avg / 60  # 5 / 60 = 0.08333...

# 4. Generowanie wektora osi X
# Oś X zaczyna się od czasu_start i trwa przez tyle kroków, ile mamy wyników PRx
czas_os_x_minuty = czas_start_minuty + np.arange(liczba_okien) * krok_minuty

# --- 5. RYSUJ INTERAKTYWNY WYKRES (ZAMIAST MATPLOTLIB) ---
fig = go.Figure()

# Dodanie serii danych PRx ze zdefiniowaną osią czasu w minutach
fig.add_trace(go.Scatter(
    x=czas_os_x_minuty,
    y=wyniki_prx,
    mode='lines',
    name='PRx (Korelacja ICP/ABP)',
    line=dict(color='blue', width=2),
    # Precyzyjne formatowanie dymka (hover) po najechaniu myszką
    hovertemplate='<b>Czas:</b> %{x:.2f} min<br><b>PRx:</b> %{y:.3f}<extra></extra>'
))

# Konfiguracja osi, siatki oraz interakcji (zoom/pan)
fig.update_layout(
    title=dict(
        text='Monitorowanie indeksu PRx w czasie',
        x=0.5,
        font=dict(size=16)
    ),
    xaxis=dict(
        title='Czas (minuty)',
        gridcolor='rgba(200, 200, 200, 0.2)',
        showspikes=True,       # Pionowa linia pomocnicza śledząca kursor
        spikemode='across',
        spikethickness=1,
        spikedash='dash'
    ),
    yaxis=dict(
        title='Wartość PRx',
        range=[-1.1, 1.1],     # Sztywny zakres dla współczynnika korelacji Pearsona
        gridcolor='rgba(200, 200, 200, 0.2)',
        zeroline=True,
        zerolinecolor='rgba(0, 0, 0, 0.3)',
        zerolinewidth=1
    ),
    template='plotly_white',
    hovermode='x unified',     # Łączenie etykiet danych dla tej samej wartości osi X
    dragmode='zoom',           # Domyślne działanie myszy to zaznaczanie obszaru (zoom)
    width=1600,                 # Odpowiednik figsize=(10, 4)
    height=400
)

# Wyświetlenie wykresu (w Jupyter Notebook pojawi się w komórce, w skrypcie .py otworzy przeglądarkę)
fig.show()

In [ ]:
x_seconds_avg = 15
window_size = 20

okna_icp, okna_abp = processor.get_windowed_data(x_seconds_avg=x_seconds_avg, window_size=window_size)
wyniki_prx = calculate_PRx(okna_icp, okna_abp)

liczba_okien = len(wyniki_prx)

czas_start_minuty = (window_size * x_seconds_avg) / 60  # Dla 60 i 5 to będzie równo 5.0 min

# Krok czasowy między kolejnymi oknami w minutach
krok_minuty = x_seconds_avg / 60  # 5 / 60 = 0.08333...

# 4. Generowanie wektora osi X
# Oś X zaczyna się od czasu_start i trwa przez tyle kroków, ile mamy wyników PRx
czas_os_x_minuty = czas_start_minuty + np.arange(liczba_okien) * krok_minuty

# --- 5. RYSUJ INTERAKTYWNY WYKRES (ZAMIAST MATPLOTLIB) ---
fig = go.Figure()

# Dodanie serii danych PRx ze zdefiniowaną osią czasu w minutach
fig.add_trace(go.Scatter(
    x=czas_os_x_minuty,
    y=wyniki_prx,
    mode='lines',
    name='PRx (Korelacja ICP/ABP)',
    line=dict(color='blue', width=2),
    # Precyzyjne formatowanie dymka (hover) po najechaniu myszką
    hovertemplate='<b>Czas:</b> %{x:.2f} min<br><b>PRx:</b> %{y:.3f}<extra></extra>'
))

# Konfiguracja osi, siatki oraz interakcji (zoom/pan)
fig.update_layout(
    title=dict(
        text='Monitorowanie indeksu PRx w czasie',
        x=0.5,
        font=dict(size=16)
    ),
    xaxis=dict(
        title='Czas (minuty)',
        gridcolor='rgba(200, 200, 200, 0.2)',
        showspikes=True,       # Pionowa linia pomocnicza śledząca kursor
        spikemode='across',
        spikethickness=1,
        spikedash='dash'
    ),
    yaxis=dict(
        title='Wartość PRx',
        range=[-1.1, 1.1],     # Sztywny zakres dla współczynnika korelacji Pearsona
        gridcolor='rgba(200, 200, 200, 0.2)',
        zeroline=True,
        zerolinecolor='rgba(0, 0, 0, 0.3)',
        zerolinewidth=1
    ),
    template='plotly_white',
    hovermode='x unified',     # Łączenie etykiet danych dla tej samej wartości osi X
    dragmode='zoom',           # Domyślne działanie myszy to zaznaczanie obszaru (zoom)
    width=1600,                 # Odpowiednik figsize=(10, 4)
    height=400
)

# Wyświetlenie wykresu (w Jupyter Notebook pojawi się w komórce, w skrypcie .py otworzy przeglądarkę)
fig.show()